# Rap-Snacks ColabFold Validation

**Two sections — run in separate sessions or sequentially:**
- **Section A** — ProteinMPNN × temps [0.1, 0.2, 0.3] + ColabFold self-consistency
- **Section B** — Originals (concordance + native_ala) + Scrambled → ColabFold (apples-to-apples)

**Setup:** All inputs/outputs live in Google Drive at `MyDrive/rap_snacks/`.  
**Runtime:** A100 GPU. T4 works but ~3× slower.

---
## 0 — Install, Mount Drive, Setup
Run every session.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'colabfold[alphafold]'], check=True)

import os
if not os.path.exists('/content/ProteinMPNN'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/dauparas/ProteinMPNN.git',
                    '/content/ProteinMPNN'], check=True)
    print('ProteinMPNN cloned')
else:
    print('ProteinMPNN already present')
print('Install complete.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── Drive folder layout ──────────────────────────────────────────────
# MyDrive/rap_snacks/
#   inputs/                   ← upload your zip here and extract once
#     data/phase2_candidates.csv
#     data/phase2_fixed_positions.jsonl
#     data/aggregated_lines_v2_enriched.csv
#     data/boltz_id_map.json
#     outputs/boltz/boltz_summary.csv
#     outputs/masks/mask_v2_concordance.json
#     outputs/proteinmpnn/pdbs/*.pdb
#   results/                  ← outputs written here (persist across sessions)
#     mpnn_colabfold_results.csv
#     mpnn_colabfold_passing.fasta
#     scrambled_colabfold_results.csv
#     figures/
# ─────────────────────────────────────────────────────────────────────

from pathlib import Path
DRIVE_ROOT  = Path('/content/drive/MyDrive/rap_snacks')
DRIVE_IN    = DRIVE_ROOT / 'inputs'
DRIVE_RES   = DRIVE_ROOT / 'results'
DRIVE_FIGS  = DRIVE_RES  / 'figures'

for d in [DRIVE_IN, DRIVE_RES, DRIVE_FIGS]:
    d.mkdir(parents=True, exist_ok=True)

# Working scratch (fast local SSD — ColabFold writes lots of temp files)
CONTENT     = Path('/content/scratch')
CONTENT.mkdir(exist_ok=True)

print(f'Drive mounted. Input folder: {DRIVE_IN}')
print(f'Results folder:             {DRIVE_RES}')

### First-time setup — upload inputs to Drive

Run this **once only**. After that, inputs persist in Drive across all sessions.

```bash
# Pack locally from rap-snacks-v1/ directory:
zip rap_snacks_inputs.zip \
    data/phase2_candidates.csv \
    data/phase2_fixed_positions.jsonl \
    data/aggregated_lines_v2_enriched.csv \
    data/boltz_id_map.json \
    outputs/boltz/boltz_summary.csv \
    outputs/masks/mask_v2_concordance.json \
    outputs/proteinmpnn/pdbs/*.pdb
```

Then upload `rap_snacks_inputs.zip` to `MyDrive/rap_snacks/inputs/` via the Drive UI,  
and run the extraction cell below.

In [ ]:
# ── Run once to extract the uploaded zip into Drive/inputs/ ──
# Skip this cell if inputs are already extracted.
import zipfile

zip_path = DRIVE_IN / 'rap_snacks_inputs.zip'
if zip_path.exists():
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DRIVE_IN)
    print(f'Extracted {zip_path.name}')
else:
    print('No zip found — inputs should already be extracted, or upload the zip first.')

# Verify
for p in [
    'data/phase2_candidates.csv',
    'data/phase2_fixed_positions.jsonl',
    'data/aggregated_lines_v2_enriched.csv',
    'outputs/boltz/boltz_summary.csv',
    'data/boltz_id_map.json',
    'outputs/masks/mask_v2_concordance.json',
]:
    exists = (DRIVE_IN / p).exists()
    print(f"  [{'OK' if exists else 'MISSING'}] {p}")
pdb_count = len(list((DRIVE_IN / 'outputs/proteinmpnn/pdbs').glob('*.pdb')))
print(f'  PDBs: {pdb_count} found')

---
## Config (shared — edit here)

In [ ]:
import json, random, subprocess, sys, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

MPNN_SCRIPT  = Path('/content/ProteinMPNN/protein_mpnn_run.py')
MPNN_WEIGHTS = Path('/content/ProteinMPNN/vanilla_model_weights')

# ── MPNN ──
TOP_N       = 12
N_SEQS      = 50
TEMPS       = [0.1, 0.2, 0.3]
PLDDT_MIN   = 0.35

# ── Scrambled control ──
N_SHUFFLES  = 3
SEED        = 42

# ── ColabFold ──
NUM_RECYCLE = 1      # 1 for filtering; set 3 for final submission sequences
MODEL_TYPE  = 'alphafold2_ptm'
NUM_MODELS  = 1

# ── Input paths (from Drive) ──
DATA        = DRIVE_IN / 'data'
MASKS       = DRIVE_IN / 'outputs' / 'masks'
PDB_DIR     = DRIVE_IN / 'outputs' / 'proteinmpnn' / 'pdbs'
FIXED_JSONL = DATA / 'phase2_fixed_positions.jsonl'

CANDIDATES  = pd.read_csv(DATA / 'phase2_candidates.csv').head(TOP_N)
MASK_BY_BAR = {m['bar_id']: m for m in json.load(open(MASKS / 'mask_v2_concordance.json'))}

# ── Scratch (ColabFold temp output — fast SSD) ──
SCRATCH_DESIGNED = CONTENT / 'designed'
SCRATCH_CF_MPNN  = CONTENT / 'cf_mpnn_output'
SCRATCH_CF_SC    = CONTENT / 'cf_sc_output'

print(f'Candidates: {len(CANDIDATES)} bars')
print(f'MPNN weights: {MPNN_WEIGHTS.exists()}')
print(CANDIDATES[['bar_id']].to_string(index=False))

---
## Helpers (shared)

In [ ]:
def fix_boltz_chain(pdb_text):
    out = []
    for line in pdb_text.splitlines(keepends=True):
        if (line.startswith('ATOM') or line.startswith('HETATM')) and len(line) >= 32:
            line = line[:21] + 'A' + line[24:]
        out.append(line)
    return ''.join(out)


def parse_mpnn_fasta(fa_path):
    seqs, header, seq = [], None, []
    for line in Path(fa_path).read_text().splitlines():
        if line.startswith('>'):
            if header: seqs.append((header, ''.join(seq)))
            header, seq = line[1:].strip(), []
        else:
            seq.append(line.strip())
    if header: seqs.append((header, ''.join(seq)))
    return seqs


def parse_colabfold_scores(scores_dir):
    """
    Parse ColabFold output directory.
    Returns {name: {'plddt': float (0-1), 'ptm': float}}.
    ColabFold stores pLDDT as 0-100 in JSON — we divide by 100.
    """
    results = {}
    for p in sorted(Path(scores_dir).glob('*_scores_rank_001*.json')):
        name = p.name.split('_scores_rank_001')[0]
        data = json.loads(p.read_text())
        results[name] = {
            'plddt': float(np.mean(data['plddt'])) / 100.0,
            'ptm':   data.get('ptm'),
        }
    return results


def run_colabfold(fasta_path, out_dir):
    """Run colabfold_batch, return parsed scores."""
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    cmd = [
        'colabfold_batch', str(fasta_path), str(out_dir),
        '--num-recycle', str(NUM_RECYCLE),
        '--model-type',  MODEL_TYPE,
        '--num-models',  str(NUM_MODELS),
    ]
    print(f'colabfold_batch {Path(fasta_path).name} → {Path(out_dir).name} ...')
    r = subprocess.run(cmd, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'ColabFold failed (exit {r.returncode})')
    return parse_colabfold_scores(out_dir)


def shuffle_sequence(seq, rng):
    chars = list(seq)
    rng.shuffle(chars)
    return ''.join(chars)


def save_to_drive(src, dst_name, subfolder=None):
    """Copy a local file to Drive results folder."""
    dst_dir = DRIVE_FIGS if subfolder == 'figures' else DRIVE_RES
    dst = dst_dir / dst_name
    shutil.copy2(src, dst)
    print(f'  Saved → Drive: {dst.relative_to(DRIVE_ROOT)}')
    return dst


# ── Dark theme helpers ──
DARK_BG = '#0e0e0e'
WHITE   = 'white'

def dark_ax(ax):
    ax.set_facecolor(DARK_BG)
    ax.tick_params(colors=WHITE)
    for label in [ax.yaxis.label, ax.xaxis.label, ax.title]:
        label.set_color(WHITE)
    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')

print('Helpers loaded.')

---
---
# SECTION A — ProteinMPNN × 3 Temperatures + ColabFold

Runs MPNN at **0.1 / 0.2 / 0.3** for all 12 candidate bars.  
Higher temperatures let dropout bars (0/50 pass at 0.1) explore more sequence space.  
All designs folded in one ColabFold batch (~1800 seqs, A100 ≈ 45–60 min at recycle=1).  
Results written to `Drive/rap_snacks/results/`.

In [ ]:
# A1 — Verify PDBs and fixed_positions.jsonl
SCRATCH_DESIGNED.mkdir(parents=True, exist_ok=True)

fixed_pos_records = []
for _, row in CANDIDATES.iterrows():
    bar_id  = row['bar_id']
    pdb_src = PDB_DIR / f'{bar_id}.pdb'
    if not pdb_src.exists():
        print(f'  [MISSING PDB] {bar_id}')
        continue
    # Copy PDB to scratch and fix chain IDs
    pdb_local = CONTENT / f'{bar_id}.pdb'
    pdb_local.write_text(fix_boltz_chain(pdb_src.read_text()))

    m       = MASK_BY_BAR.get(bar_id, {})
    bjozxu  = set(m.get('mutation_mask', []))
    seq_len = m.get('sequence_length', int(row['fasta_seq_len']))
    fixed_pos_records.append({
        'bar_id':  bar_id,
        'fixed':   [i+1 for i in range(seq_len) if i not in bjozxu],
        'design':  [i+1 for i in sorted(bjozxu)],
        'seq_len': seq_len,
    })
    print(f'  {bar_id}  len={seq_len}  designable={len(bjozxu)}')

# Write single-object JSONL (all bars on one line — MPNN requires this format)
fixed_jsonl_local = CONTENT / 'fixed_positions.jsonl'
combined = {r['bar_id']: {'A': r['fixed']} for r in fixed_pos_records}
with open(fixed_jsonl_local, 'w') as f:
    f.write(json.dumps(combined) + '\n')
print(f'\nPrepared {len(fixed_pos_records)} bars, fixed_positions.jsonl written.')

In [ ]:
# A2 — Run ProteinMPNN at each temperature
for temp in TEMPS:
    print(f'\n====== Temperature {temp} ======')
    for rec in fixed_pos_records:
        bar_id  = rec['bar_id']
        pdb_in  = CONTENT / f'{bar_id}.pdb'
        out_dir = SCRATCH_DESIGNED / f't{temp}' / bar_id
        out_dir.mkdir(parents=True, exist_ok=True)

        cmd = [
            sys.executable, str(MPNN_SCRIPT),
            '--pdb_path',              str(pdb_in),
            '--out_folder',            str(out_dir),
            '--num_seq_per_target',    str(N_SEQS),
            '--sampling_temp',         str(temp),
            '--fixed_positions_jsonl', str(fixed_jsonl_local),
            '--path_to_model_weights', str(MPNN_WEIGHTS),
            '--batch_size',            '1',
            '--suppress_print',        '1',
        ]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'  [ERROR] {bar_id}: {r.stderr[-300:]}')
            continue
        fa = list(out_dir.rglob('*.fa')) + list(out_dir.rglob('*.fasta'))
        n  = len(parse_mpnn_fasta(fa[0])) - 1 if fa else 0
        print(f'  {bar_id}  temp={temp}  {n} designs')

print('\nMPNN complete.')

In [ ]:
# A3 — Collect all designs into one FASTA
mpnn_fasta_local = CONTENT / 'mpnn_designs_all_temps.fasta'
design_meta      = []

with open(mpnn_fasta_local, 'w') as fout:
    for temp in TEMPS:
        for rec in fixed_pos_records:
            bar_id  = rec['bar_id']
            out_dir = SCRATCH_DESIGNED / f't{temp}' / bar_id
            fa = list(out_dir.rglob('*.fa')) + list(out_dir.rglob('*.fasta'))
            if not fa: continue
            ts = str(temp).replace('.', 'p')
            for i, (header, seq) in enumerate(parse_mpnn_fasta(fa[0])[1:]):
                cf_name = f'{bar_id}_t{ts}_d{i:02d}'
                fout.write(f'>{cf_name}\n{seq}\n')
                design_meta.append({'cf_name': cf_name, 'bar_id': bar_id,
                                     'temp': temp, 'design_idx': i,
                                     'header': header, 'sequence': seq,
                                     'seq_len': len(seq)})

meta_df = pd.DataFrame(design_meta)
print(f'Total designs: {len(meta_df)}')
print(meta_df.groupby(['bar_id','temp']).size().unstack(fill_value=0))

In [ ]:
# A4 — Run ColabFold (all temps in one batch)
# ~1800 seqs on A100 ≈ 45–60 min at num-recycle=1
scores = run_colabfold(mpnn_fasta_local, SCRATCH_CF_MPNN)
print(f'Scored {len(scores)} sequences.')

In [ ]:
# A5 — Merge, filter, save to Drive
meta_df['cf_plddt'] = meta_df['cf_name'].map(lambda n: scores.get(n, {}).get('plddt'))
meta_df['cf_ptm']   = meta_df['cf_name'].map(lambda n: scores.get(n, {}).get('ptm'))
meta_df['passes']   = meta_df['cf_plddt'].apply(lambda p: p is not None and p >= PLDDT_MIN)

# Save CSV
mpnn_csv_local = CONTENT / 'mpnn_colabfold_results.csv'
meta_df.to_csv(mpnn_csv_local, index=False)
save_to_drive(mpnn_csv_local, 'mpnn_colabfold_results.csv')

# Save passing FASTA
passing = meta_df[meta_df['passes']]
mpnn_pass_local = CONTENT / 'mpnn_colabfold_passing.fasta'
with open(mpnn_pass_local, 'w') as f:
    for _, row in passing.iterrows():
        f.write(f">{row['bar_id']}_t{row['temp']}_d{row['design_idx']:02d} | {row['header']}\n")
        f.write(f"{row['sequence']}\n")
save_to_drive(mpnn_pass_local, 'mpnn_colabfold_passing.fasta')

# Summary
summary = meta_df.groupby(['bar_id','temp']).agg(
    n=('passes','count'), n_pass=('passes','sum'), mean_plddt=('cf_plddt','mean')
).reset_index()
summary['pass_rate'] = summary['n_pass'] / summary['n']
print(f'Passing: {len(passing)} / {len(meta_df)}')
print('\n--- Per bar per temp ---')
print(summary.to_string(index=False))

In [ ]:
# A6 — Figures → Drive
bar_ids      = sorted(meta_df['bar_id'].unique())
temps        = sorted(meta_df['temp'].unique())
temp_colors  = {0.1: '#00d4ff', 0.2: '#ff9500', 0.3: '#cc44ff'}
DROPOUT_BARS = ['bar_0', 'bar_8', 'bar_46']

# ── Fig A1: Pass rate by bar × temp ──
fig, ax = plt.subplots(figsize=(14, 5), facecolor=DARK_BG)
dark_ax(ax)
x, w = np.arange(len(bar_ids)), 0.25
for j, temp in enumerate(temps):
    sub  = summary[summary['temp']==temp].set_index('bar_id').reindex(bar_ids)
    ax.bar(x+(j-1)*w, sub['pass_rate'].fillna(0), w,
           label=f'temp={temp}', color=temp_colors[temp], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(bar_ids, rotation=45, ha='right', color=WHITE, fontsize=8)
ax.set_ylabel('Pass rate (pLDDT >= 0.35)', color=WHITE)
ax.set_ylim(0, 1.05)
ax.set_title('ProteinMPNN Pass Rate by Bar x Temperature (ColabFold)', color=WHITE)
ax.legend(facecolor='#1a1a1a', labelcolor=WHITE, fontsize=9)
plt.tight_layout()
f1 = CONTENT / 'fig_A1_pass_rate.png'
plt.savefig(f1, dpi=150, facecolor=DARK_BG); plt.show()
save_to_drive(f1, 'fig_A1_pass_rate.png', 'figures')

# ── Fig A2: pLDDT violin per bar × temp ──
fig, axes = plt.subplots(1, len(bar_ids), figsize=(2.1*len(bar_ids), 5), facecolor=DARK_BG)
if len(bar_ids) == 1: axes = [axes]
for ax, bar_id in zip(axes, bar_ids):
    dark_ax(ax)
    for j, temp in enumerate(temps):
        vals = meta_df[(meta_df['bar_id']==bar_id)&(meta_df['temp']==temp)]['cf_plddt'].dropna()
        if len(vals) < 2: continue
        vp = ax.violinplot(vals, positions=[j], widths=0.6, showmedians=True)
        for part in vp['bodies']:
            part.set_facecolor(temp_colors[temp]); part.set_alpha(0.7)
        vp['cmedians'].set_color(WHITE)
    ax.axhline(PLDDT_MIN, color='#ff4444', lw=0.8, linestyle='--')
    ax.set_xticks(range(len(temps)))
    ax.set_xticklabels([str(t) for t in temps], fontsize=7, color=WHITE)
    ax.set_ylim(0, 1)
    ax.set_title(bar_id, fontsize=8, color=WHITE)
    if bar_id == bar_ids[0]: ax.set_ylabel('ColabFold pLDDT', color=WHITE)
plt.suptitle('pLDDT Distribution per Bar x Temperature', color=WHITE, fontsize=11, y=1.01)
plt.tight_layout()
f2 = CONTENT / 'fig_A2_plddt_violin.png'
plt.savefig(f2, dpi=150, facecolor=DARK_BG, bbox_inches='tight'); plt.show()
save_to_drive(f2, 'fig_A2_plddt_violin.png', 'figures')

# ── Fig A3: Dropout bar recovery ──
dropout_df = meta_df[meta_df['bar_id'].isin(DROPOUT_BARS)]
if len(dropout_df):
    fig, ax = plt.subplots(figsize=(9, 4), facecolor=DARK_BG)
    dark_ax(ax)
    xtick_labels = []
    xi = 0
    for bar_id in DROPOUT_BARS:
        for temp in temps:
            vals = dropout_df[(dropout_df['bar_id']==bar_id)&(dropout_df['temp']==temp)]['cf_plddt'].dropna()
            ax.scatter([xi]*len(vals), vals, color=temp_colors[temp], s=15, alpha=0.5)
            if len(vals): ax.scatter(xi, vals.mean(), color=WHITE, s=50, zorder=5, marker='D')
            xtick_labels.append(f'{bar_id}\nt={temp}')
            xi += 1
        xi += 0.5  # gap between bars
    ax.axhline(PLDDT_MIN, color='#ff4444', lw=1, linestyle='--', label=f'threshold {PLDDT_MIN}')
    ax.set_xticks(range(len(xtick_labels)))
    ax.set_xticklabels(xtick_labels, rotation=30, ha='right', color=WHITE, fontsize=7)
    ax.set_ylabel('ColabFold pLDDT', color=WHITE)
    ax.set_title('Dropout Bar Recovery Across Temperatures', color=WHITE)
    ax.legend(facecolor='#1a1a1a', labelcolor=WHITE)
    plt.tight_layout()
    f3 = CONTENT / 'fig_A3_dropout_recovery.png'
    plt.savefig(f3, dpi=150, facecolor=DARK_BG); plt.show()
    save_to_drive(f3, 'fig_A3_dropout_recovery.png', 'figures')

print('Section A figures saved to Drive.')

---
---
# SECTION B — Originals + Scrambled → ColabFold (Apples-to-Apples)

**Run in a fresh session** (or after Section A finishes) to free GPU memory.

All four buckets are folded with the **same ColabFold model** in a single batch:
- `concordance` — original lyric-derived sequence
- `native_ala` — alanine substitution at BJOZXU positions
- `scrambled_concordance` — concordance shuffled (same AA composition, random order)
- `scrambled_native_ala` — native_ala shuffled

This gives a fair comparison: ESMFold showed no discrimination between originals and
scrambles. ColabFold (AF2 + MSA) is order-sensitive and may show a real signal.

In [ ]:
# B1 — Load sequences and generate scrambles
enriched = pd.read_csv(DATA / 'aggregated_lines_v2_enriched.csv')
cands    = pd.read_csv(DATA / 'phase2_candidates.csv')
enriched = enriched[enriched['bar_id'].isin(set(cands['bar_id']))]
enriched = enriched.dropna(subset=['fasta_seq_concordance']).reset_index(drop=True)
print(f'{len(enriched)} candidate bars with concordance sequence')

rng  = random.Random(SEED)
rows = []

for _, row in enriched.iterrows():
    bar_id   = row['bar_id']
    conc_seq = str(row.get('fasta_seq_concordance', '') or '')
    na_seq   = str(row.get('fasta_seq_native_alanine', '') or '')

    if conc_seq:
        rows.append({'bar_id': bar_id, 'source': 'concordance',
                     'shuffle_idx': -1, 'sequence': conc_seq, 'seq_len': len(conc_seq)})
    if na_seq:
        rows.append({'bar_id': bar_id, 'source': 'native_ala',
                     'shuffle_idx': -1, 'sequence': na_seq,   'seq_len': len(na_seq)})
    for i in range(N_SHUFFLES):
        if conc_seq:
            rows.append({'bar_id': bar_id, 'source': 'scrambled_concordance',
                         'shuffle_idx': i, 'sequence': shuffle_sequence(conc_seq, rng), 'seq_len': len(conc_seq)})
        if na_seq:
            rows.append({'bar_id': bar_id, 'source': 'scrambled_native_ala',
                         'shuffle_idx': i, 'sequence': shuffle_sequence(na_seq, rng),   'seq_len': len(na_seq)})

sc_df = pd.DataFrame(rows)

# ColabFold sequence names: short, no special chars
sc_df['cf_name'] = (
    sc_df['bar_id'] + '_' +
    sc_df['source']
        .str.replace('scrambled_concordance', 'scconc')
        .str.replace('scrambled_native_ala',  'scna')
        .str.replace('concordance', 'conc')
        .str.replace('native_ala',  'na') +
    '_' + sc_df['shuffle_idx'].astype(str)
)

print(f'Total sequences: {len(sc_df)}')
print(sc_df.groupby('source').size())

In [ ]:
# B2 — Write single FASTA (originals + scrambles)
sc_fasta_local = CONTENT / 'originals_and_scrambled.fasta'
with open(sc_fasta_local, 'w') as f:
    for _, row in sc_df.iterrows():
        f.write(f">{row['cf_name']}\n{row['sequence']}\n")
print(f'Wrote {len(sc_df)} sequences → {sc_fasta_local.name}')

In [ ]:
# B3 — Run ColabFold (originals + scrambles in one batch)
# ~12 bars x (2 orig + 6 scrambles) = ~96 seqs on A100 ≈ 5-8 min
sc_scores = run_colabfold(sc_fasta_local, SCRATCH_CF_SC)
print(f'Scored {len(sc_scores)} sequences.')

In [ ]:
# B4 — Merge scores and save to Drive
sc_df['cf_plddt'] = sc_df['cf_name'].map(lambda n: sc_scores.get(n, {}).get('plddt'))
sc_df['cf_ptm']   = sc_df['cf_name'].map(lambda n: sc_scores.get(n, {}).get('ptm'))

sc_csv_local = CONTENT / 'scrambled_colabfold_results.csv'
sc_df.to_csv(sc_csv_local, index=False)
save_to_drive(sc_csv_local, 'scrambled_colabfold_results.csv')

print('\n--- pLDDT by bucket (ColabFold) ---')
for src in ['concordance', 'native_ala', 'scrambled_concordance', 'scrambled_native_ala']:
    vals = sc_df.loc[sc_df['source']==src, 'cf_plddt'].dropna()
    if len(vals):
        print(f'  {src:25s}  n={len(vals):3d}  mean={vals.mean():.3f}  '
              f'sd={vals.std():.3f}  <0.3: {(vals<0.3).sum()}/{len(vals)}')

In [ ]:
# B5 — Figures → Drive
COLORS = {
    'concordance':           '#00d4ff',
    'native_ala':            '#ff9500',
    'scrambled_concordance': '#444455',
    'scrambled_native_ala':  '#664444',
}
LABELS = {
    'concordance':           'concordance (ColabFold)',
    'native_ala':            'native_ala (ColabFold)',
    'scrambled_concordance': 'scrambled concordance',
    'scrambled_native_ala':  'scrambled native_ala',
}
sources_ord = ['concordance', 'native_ala', 'scrambled_concordance', 'scrambled_native_ala']
bar_ids     = list(sc_df['bar_id'].unique())
x_pos       = {b: i for i, b in enumerate(bar_ids)}

# ── Fig B1: per-bar scatter (4 sources) ──
fig, ax = plt.subplots(figsize=(max(10, len(bar_ids)*0.9), 5), facecolor=DARK_BG)
dark_ax(ax)
plotted = set()
for _, row in sc_df.iterrows():
    plddt = row['cf_plddt']
    if plddt is None or pd.isna(plddt): continue
    src     = row['source']
    is_orig = row['shuffle_idx'] == -1
    label   = LABELS.get(src, src) if src not in plotted else ''
    ax.scatter(x_pos[row['bar_id']], plddt,
               color=COLORS.get(src,'#888'),
               s=70 if is_orig else 20,
               marker='D' if is_orig else 'o',
               alpha=0.85, zorder=4 if is_orig else 2, label=label)
    plotted.add(src)
ax.axhline(0.3, color='#ff4444', lw=1, linestyle='--', label='pLDDT=0.3')
ax.set_xticks(range(len(bar_ids)))
ax.set_xticklabels(bar_ids, rotation=45, ha='right', fontsize=7, color=WHITE)
ax.set_ylabel('ColabFold pLDDT', color=WHITE)
ax.set_ylim(0, 1)
ax.set_title('Concordance vs Native_Ala vs Scrambled (ColabFold, apples-to-apples)', color=WHITE)
handles, labels_ = ax.get_legend_handles_labels()
seen = {}
for h, l in zip(handles, labels_):
    if l and l not in seen: seen[l] = h
ax.legend(seen.values(), seen.keys(), facecolor='#1a1a1a', labelcolor=WHITE, fontsize=8)
plt.tight_layout()
f_b1 = CONTENT / 'fig_B1_scatter.png'
plt.savefig(f_b1, dpi=150, facecolor=DARK_BG); plt.show()
save_to_drive(f_b1, 'fig_B1_scatter.png', 'figures')

# ── Fig B2: boxplot by bucket ──
fig, ax = plt.subplots(figsize=(8, 5), facecolor=DARK_BG)
dark_ax(ax)
data_by_src = [sc_df.loc[sc_df['source']==s, 'cf_plddt'].dropna().values for s in sources_ord]
bp = ax.boxplot(data_by_src, patch_artist=True, medianprops={'color': WHITE, 'lw': 2})
for patch, src in zip(bp['boxes'], sources_ord):
    patch.set_facecolor(COLORS[src]); patch.set_alpha(0.8)
for elem in ['whiskers','caps','fliers']:
    for part in bp[elem]: part.set_color('#888')
ax.axhline(0.3, color='#ff4444', lw=1, linestyle='--', label='pLDDT=0.3')
ax.set_xticks(range(1, len(sources_ord)+1))
ax.set_xticklabels([LABELS[s] for s in sources_ord], rotation=20, ha='right', color=WHITE, fontsize=8)
ax.set_ylabel('ColabFold pLDDT', color=WHITE)
ax.set_ylim(0, 1)
ax.set_title('pLDDT Distribution: Originals vs Scrambled (ColabFold)', color=WHITE)
ax.legend(facecolor='#1a1a1a', labelcolor=WHITE)
plt.tight_layout()
f_b2 = CONTENT / 'fig_B2_boxplot.png'
plt.savefig(f_b2, dpi=150, facecolor=DARK_BG); plt.show()
save_to_drive(f_b2, 'fig_B2_boxplot.png', 'figures')

# ── Fig B3: delta pLDDT per bar (original minus scrambled mean) ──
delta_rows = []
for bar_id in bar_ids:
    sub = sc_df[sc_df['bar_id']==bar_id]
    for src, sc_src in [('concordance','scrambled_concordance'), ('native_ala','scrambled_native_ala')]:
        orig_v = sub.loc[sub['source']==src,    'cf_plddt'].mean()
        sc_v   = sub.loc[sub['source']==sc_src, 'cf_plddt'].mean()
        if pd.notna(orig_v) and pd.notna(sc_v):
            delta_rows.append({'bar_id': bar_id, 'source': src, 'delta': orig_v - sc_v})
delta_df = pd.DataFrame(delta_rows)

fig, ax = plt.subplots(figsize=(max(10, len(bar_ids)*0.9), 4), facecolor=DARK_BG)
dark_ax(ax)
x = np.arange(len(bar_ids))
for j, src in enumerate(['concordance','native_ala']):
    sub = delta_df[delta_df['source']==src].set_index('bar_id').reindex(bar_ids)['delta']
    ax.bar(x+(j-0.5)*0.3, sub.values, 0.3, color=COLORS[src], alpha=0.85, label=LABELS[src])
ax.axhline(0, color='#888', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(bar_ids, rotation=45, ha='right', fontsize=7, color=WHITE)
ax.set_ylabel('pLDDT(original) - pLDDT(scrambled)', color=WHITE)
ax.set_title('Structure Signal: Original - Scrambled pLDDT per Bar\n(positive = structure is sequence-order dependent)', color=WHITE)
ax.legend(facecolor='#1a1a1a', labelcolor=WHITE, fontsize=8)
plt.tight_layout()
f_b3 = CONTENT / 'fig_B3_delta_plddt.png'
plt.savefig(f_b3, dpi=150, facecolor=DARK_BG); plt.show()
save_to_drive(f_b3, 'fig_B3_delta_plddt.png', 'figures')

print('Section B figures saved to Drive.')